# 🎬 Movie Recommendation System
### Content-Based Filtering · TMDB 5000 Movies Dataset

**Type:** Content-Based Filtering | **Stack:** Python · Pandas · Scikit-learn · NLTK

### Pipeline
```
Download → Load & Merge → Feature Engineering → Preprocess → Vectorize → Similarity → Recommend
```

## 📥 Step 0 — Download Dataset (No Kaggle Login Needed)

Using a public GitHub mirror — auto-downloads both CSV files.

In [ ]:
# Download TMDB dataset directly — no Kaggle account needed
import urllib.request, os

FILES = {
    'tmdb_5000_movies.csv':  'https://raw.githubusercontent.com/dsrscientist/dataset1/master/tmdb_5000_movies.csv',
    'tmdb_5000_credits.csv': 'https://raw.githubusercontent.com/dsrscientist/dataset1/master/tmdb_5000_credits.csv'
}

for filename, url in FILES.items():
    if os.path.exists(filename):
        print(f'  Already present: {filename}')
    else:
        print(f'  Downloading {filename}...', end=' ', flush=True)
        urllib.request.urlretrieve(url, filename)
        print(f'done ({os.path.getsize(filename)/1e6:.1f} MB)')

print('Both dataset files ready!')

## 📦 Step 1 — Import Libraries

In [ ]:
# All pre-installed in Google Colab
import pandas as pd
import numpy as np
import ast                              # parse stringified JSON columns
import pickle                           # save model for deployment
import nltk
from nltk.stem import PorterStemmer    # reduce words to root form
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.metrics.pairwise import cosine_similarity

nltk.download('punkt', quiet=True)
print(f'Libraries loaded | pandas {pd.__version__} | numpy {np.__version__}')

## 🔗 Step 2 — Load & Merge Datasets

| File | Key columns |
|---|---|
| `tmdb_5000_movies.csv` | movie_id, title, overview, genres, keywords |
| `tmdb_5000_credits.csv` | movie_id, title, cast, crew |

Merge both on `title` to create one unified DataFrame.

In [ ]:
# Load both CSV files
movies  = pd.read_csv('tmdb_5000_movies.csv')
credits = pd.read_csv('tmdb_5000_credits.csv')

print(f'Movies shape  : {movies.shape}')
print(f'Credits shape : {credits.shape}')

# Merge on title
df = movies.merge(credits, on='title')

# Keep only the 7 columns we need
df = df[['movie_id', 'title', 'overview', 'genres', 'keywords', 'cast', 'crew']]

print(f'Merged shape  : {df.shape}')
print(f'Null values:\n{df.isnull().sum()}')
df.head(3)

## ⚙️ Step 3 — Feature Engineering

Columns `genres`, `keywords`, `cast`, `crew` are stored as JSON strings.

| Column | Extract |
|---|---|
| genres | All genre names |
| keywords | All keyword names |
| cast | **Top 3** actor names only |
| crew | **Director** name only |

In [ ]:
# Helper: parse JSON string → list of names
def parse_list(obj):
    try:
        return [i['name'] for i in ast.literal_eval(obj)]
    except:
        return []

# Helper: top-3 cast members only (limits noise from minor actors)
def top3_cast(obj):
    try:
        return [i['name'] for i in ast.literal_eval(obj)[:3]]
    except:
        return []

# Helper: extract Director from crew JSON
def get_director(obj):
    try:
        return [i['name'] for i in ast.literal_eval(obj) if i.get('job') == 'Director']
    except:
        return []

# Apply helpers to DataFrame
df['genres']   = df['genres'].apply(parse_list)
df['keywords'] = df['keywords'].apply(parse_list)
df['cast']     = df['cast'].apply(top3_cast)
df['crew']     = df['crew'].apply(get_director)

print('Feature extraction done.')
df[['title', 'genres', 'cast', 'crew']].head(3)

## 🧹 Step 4 — Text Preprocessing

1. **Collapse spaces** — `Sam Worthington` → `SamWorthington` (prevents token splitting)
2. **Combine** all features into one `tags` string per movie
3. **Lowercase** — consistent token matching
4. **Porter Stemming** — `dancing/dancer/dances` → `danc` (reduces vocab noise)

In [ ]:
# 4a: Collapse spaces so multi-word names become single tokens
def collapse_spaces(lst):
    return [w.replace(' ', '') for w in lst]

df['genres']   = df['genres'].apply(collapse_spaces)
df['keywords'] = df['keywords'].apply(collapse_spaces)
df['cast']     = df['cast'].apply(collapse_spaces)
df['crew']     = df['crew'].apply(collapse_spaces)

# 4b: Split overview text into word list
df['overview'] = df['overview'].fillna('').apply(str.split)

# 4c: Combine all feature lists into one tags string
df['tags'] = df['overview'] + df['genres'] + df['keywords'] + df['cast'] + df['crew']
df['tags'] = df['tags'].apply(lambda x: ' '.join(x).lower())

# 4d: Apply Porter Stemming
ps = PorterStemmer()
def stem_text(text):
    return ' '.join(ps.stem(w) for w in text.split())

df['tags'] = df['tags'].apply(stem_text)

# Build final clean DataFrame
final_df = df[['movie_id', 'title', 'tags']].reset_index(drop=True)

print(f'Preprocessing done. Shape: {final_df.shape}')
mask = final_df['title'] == 'Avatar'
if mask.any():
    print('Sample tags (Avatar):', final_df.loc[mask, 'tags'].values[0][:300], '...')

## 📊 Step 5 — Count Vectorization

Convert `tags` text to a numerical matrix. CountVectorizer chosen over TF-IDF because common genre words like `action` are *useful signal* in recommendations, not noise.

**Output:** Sparse matrix · shape `(4803 × 5000)`

In [ ]:
# Fit CountVectorizer — 5000 most common words, drop English stop words
cv = CountVectorizer(max_features=5000, stop_words='english')
vectors = cv.fit_transform(final_df['tags'])

print(f'Matrix shape : {vectors.shape}')
print(f'Memory       : {vectors.data.nbytes / 1e6:.2f} MB (sparse)')
print('Sample vocab:', cv.get_feature_names_out()[:20])

## 📐 Step 6 — Cosine Similarity Matrix

| Score | Meaning |
|---|---|
| 1.0 | Identical features — very similar |
| 0.5 | Moderate shared features |
| 0.0 | No shared features |

Output: **(4803 × 4803)** matrix where `similarity[i][j]` = similarity of movie i vs j.

In [ ]:
# Compute pairwise cosine similarity matrix
print('Computing similarity matrix...')
similarity = cosine_similarity(vectors)

print(f'Shape  : {similarity.shape}')
print(f'Memory : {similarity.nbytes / 1e6:.1f} MB')
print(f'Sanity check — movie[0] vs itself: {similarity[0][0]:.1f} (should be 1.0)')

## 🎯 Step 7 — Recommendation Function

Find movie → get similarity scores → sort descending → skip self → return top-N titles

In [ ]:
def recommend(movie_title, n=10):
    """
    Return top-n most similar movies for a given title.
    Parameters: movie_title (str, case-insensitive), n (int, default 10)
    Returns: list of movie title strings sorted by similarity
    """
    matches = final_df[final_df['title'].str.lower() == movie_title.lower()]
    if matches.empty:
        print(f"'{movie_title}' not found. Check spelling.")
        return []
    idx = matches.index[0]
    scores = sorted(enumerate(similarity[idx]), key=lambda x: x[1], reverse=True)[1:n+1]
    return [final_df.iloc[i[0]]['title'] for i in scores]


def show_recommendations(movie_title, n=10):
    """Pretty-print the top recommendations for a movie."""
    sep = '-' * 52
    print(f'\n{sep}')
    print(f'  Because you liked: {movie_title}')
    print(sep)
    for rank, title in enumerate(recommend(movie_title, n), 1):
        print(f'  {rank:>2}.  {title}')
    print(f'{sep}\n')

print('Recommendation engine ready!')

## 🧪 Step 8 — Test with 4 Example Movies

In [ ]:
show_recommendations('Avatar')

In [ ]:
show_recommendations('The Dark Knight')

In [ ]:
show_recommendations('Interstellar')

In [ ]:
show_recommendations('The Lion King')

## 🎛️ Step 9 — Interactive Widget (Colab)

Live searchable dropdown to recommend movies interactively.

In [ ]:
import ipywidgets as widgets
from IPython.display import display, clear_output

movie_input = widgets.Combobox(
    placeholder='Type a movie title...',
    options=list(final_df['title'].values),
    description='Movie:',
    layout=widgets.Layout(width='420px')
)
n_slider = widgets.IntSlider(value=10, min=5, max=20, description='Top N:')
button   = widgets.Button(description='Get Recommendations', button_style='success')
output   = widgets.Output()

def on_click(b):
    with output:
        clear_output()
        show_recommendations(movie_input.value, n=n_slider.value)

button.on_click(on_click)
display(widgets.VBox([movie_input, n_slider, button, output]))

## 💾 Step 10 — Save Model Artifacts

Pickle two files for the Streamlit web app:
- `movies.pkl` — title + movie_id DataFrame
- `similarity.pkl` — cosine similarity matrix

In [ ]:
# Save artifacts for deployment
pickle.dump(final_df[['movie_id', 'title']], open('movies.pkl', 'wb'))
pickle.dump(similarity, open('similarity.pkl', 'wb'))
print('Saved: movies.pkl and similarity.pkl')

# Auto-download in Colab
try:
    from google.colab import files
    files.download('movies.pkl')
    files.download('similarity.pkl')
    print('Download started.')
except ImportError:
    print('(Not in Colab — files saved locally.)')

## 📊 Summary

| Step | Technique | Output |
|---|---|---|
| Load & Merge | `pd.read_csv` + `.merge()` | 4,803 × 7 DataFrame |
| Feature Eng. | JSON parse, top-3 cast, director | Python lists |
| Preprocessing | Space collapse, lowercase, stemming | `tags` string column |
| Vectorization | `CountVectorizer(max_features=5000)` | 4803 × 5000 sparse matrix |
| Similarity | `cosine_similarity` | 4803 × 4803 float matrix |
| Recommend | Index lookup + sort | Top-10 movie titles |

---

**Deploy:** `streamlit run app.py` | **Free hosting:** [share.streamlit.io](https://share.streamlit.io)